<a href="https://colab.research.google.com/github/mohanraj50115-svg/BioAgentLab/blob/main/Another_copy_of_MULTI_AGENT_SYSTEM1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
!pip install streamlit pyngrok langchain langchain-google-genai faiss-cpu pymupdf

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="models/gemini-flash-latest"
)

response = llm.invoke("Hello")
print(response.content)

[{'type': 'text', 'text': 'Hello! How can I help you today?', 'extras': {'signature': 'Eq4CCqsCAQw51sdAjhF8GWW93pnKZcXU918/wo2uOLvTVxngkP6LRo/t16qFejDe4LSe6ts8Q5vb+TJUHcnurOAOhexYvXgS/2eVGkHnnLZ8yyJUo1g0R0i4ge3swqyU45rlXaa8rvlPYjQpywD9TdXBatSVcd1QjDsPWBeEplWFwzVy+6z98Nt9xA6jb55nTBKfyCUgH/U4Cj8Tz7PlohfrLXxLHTVcHwzIvMaP170JOFITC/Do20R327zGSDHKo+auyEBP+BVLpAXMD7RnTmedNe+hBVLL1KQ9UIYtKMFC7MuUAtsvzR6cgVpuwlHnuGJXfv7/+gddfP3cr0VeqOnjAFea5CZbiIVct+3CCdOMGRBjYk+TpfenIQJxSERsVrXtbnDymA7HKnAmMSGGkeE='}}]


In [1]:
%%capture
!pip install streamlit pyngrok langchain langchain-community langchain-google-genai faiss-cpu sentence-transformers pymupdf python-pptx reportlab

In [2]:
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass("Enter app key ")

Enter app key ··········


In [ ]:
%%writefile app.py
import streamlit as st
import os
import sqlite3
import hashlib
import re

# ---------- LLM ----------
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# ---------- PAGE ----------
st.set_page_config(page_title="🧬 AI Research System", layout="wide")

# ---------- DATABASE ----------
conn = sqlite3.connect("app.db", check_same_thread=False)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS users (
    username TEXT PRIMARY KEY,
    password TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS chats (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    username TEXT,
    role TEXT,
    message TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS profile (
    username TEXT,
    name TEXT
)
""")

conn.commit()

# ---------- SECURITY ----------
def hash_password(pw):
    return hashlib.sha256(pw.encode()).hexdigest()

def signup(u, p):
    try:
        cursor.execute("INSERT INTO users VALUES (?, ?)", (u, hash_password(p)))
        conn.commit()
        return True
    except:
        return False

def login(u, p):
    cursor.execute("SELECT * FROM users WHERE username=? AND password=?", (u, hash_password(p)))
    return cursor.fetchone()

# ---------- SESSION ----------
if "user" not in st.session_state:
    st.session_state.user = None

# ---------- LOGIN UI ----------
if st.session_state.user is None:
    st.title("🔐 Login / Signup")

    tab1, tab2 = st.tabs(["Login", "Signup"])

    with tab1:
        u = st.text_input("Username")
        p = st.text_input("Password", type="password")
        if st.button("Login"):
            if login(u, p):
                st.session_state.user = u
                st.success("Logged in")
                st.rerun()
            else:
                st.error("Invalid credentials")

    with tab2:
        u = st.text_input("New Username")
        p = st.text_input("New Password", type="password")
        if st.button("Create Account"):
            if signup(u, p):
                st.success("Account created")
            else:
                st.error("Username exists")

    st.stop()

# ---------- MODEL ----------
llm = ChatGoogleGenerativeAI(
    model="models/gemini-flash-latest",
    temperature=0.3
)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# ---------- CLEAN ----------
def clean_response(res):
    text = res.content if hasattr(res, "content") else str(res)
    if isinstance(text, list):
        text = " ".join([str(i) for i in text])
    text = str(text)
    text = re.sub(r'signature.*', '', text, flags=re.I)
    return text.strip()

def run_llm(prompt):
    return clean_response(llm.invoke(prompt))

# ---------- PROFILE ----------
def get_name():
    cursor.execute("SELECT name FROM profile WHERE username=?", (st.session_state.user,))
    r = cursor.fetchone()
    return r[0] if r else st.session_state.user

def save_name(name):
    cursor.execute("DELETE FROM profile WHERE username=?", (st.session_state.user,))
    cursor.execute("INSERT INTO profile VALUES (?, ?)", (st.session_state.user, name))
    conn.commit()

# ---------- CHAT MEMORY ----------
def save_chat(role, msg):
    cursor.execute("INSERT INTO chats (username, role, message) VALUES (?, ?, ?)",
                   (st.session_state.user, role, msg))
    conn.commit()

def load_chat():
    cursor.execute("SELECT role, message FROM chats WHERE username=?", (st.session_state.user,))
    return cursor.fetchall()

def clear_chat():
    cursor.execute("DELETE FROM chats WHERE username=?", (st.session_state.user,))
    conn.commit()

# ---------- PDF ----------
def process_pdf(file):
    with open("temp.pdf", "wb") as f:
        f.write(file.read())

    loader = PyMuPDFLoader("temp.pdf")
    docs = loader.load()

    splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=200)
    chunks = splitter.split_documents(docs)

    return FAISS.from_documents(chunks, embeddings)

def retrieval(vector, q):
    docs = vector.as_retriever().invoke(q)
    return "\n\n".join([d.page_content for d in docs])

# ---------- AGENT ----------
def mode(q):
    q = q.lower()
    if "experiment" in q:
        return "experiment"
    elif "proposal" in q:
        return "proposal"
    elif "gap" in q:
        return "research"
    elif "paper" in q:
        return "paper"
    elif "pdf" in q:
        return "rag"
    else:
        return "explain"

# ---------- SIDEBAR ----------
st.sidebar.write(f"👤 {get_name()}")

if st.sidebar.button("Logout"):
    st.session_state.user = None
    st.rerun()

page = st.sidebar.radio("Menu", ["Chat", "Paper Analyzer", "Profile", "Help", "About"])

pdf = st.sidebar.file_uploader("Upload PDF")

if pdf:
    st.session_state.vector = process_pdf(pdf)
    st.sidebar.success("PDF ready")

if st.sidebar.button("Clear Chat"):
    clear_chat()
    st.rerun()

# ---------- CHAT ----------
if page == "Chat":
    st.title(f"💬 Welcome {get_name()}")

    chat = load_chat()
    for r, m in chat:
        with st.chat_message(r):
            st.write(m)

    q = st.chat_input("Ask...")
    if q:
        save_chat("user", q)
        with st.chat_message("user"):
            st.write(q)

        context = ""
        if "vector" in st.session_state:
            context = retrieval(st.session_state.vector, q)

        history = "\n".join([f"{r}:{m}" for r, m in chat[-5:]])

        prompt = f"""
User: {get_name()}
History:
{history}
Context:
{context}
Question:
{q}
Answer in paragraph form.
"""

        ans = run_llm(prompt)

        save_chat("assistant", ans)

        with st.chat_message("assistant"):
            st.write(ans)

# ---------- PAPER ----------
elif page == "Paper Analyzer":
    st.title("📄 Paper Analyzer")
    if "vector" in st.session_state:
        ctx = retrieval(st.session_state.vector, "summarize paper")
        ans = run_llm(f"Explain paper clearly:\n{ctx}")
        st.write(ans)
    else:
        st.warning("Upload PDF")


# ---------- HELP ----------
elif page == "Help":
    st.write("""
Upload PDF → Ask questions
Use chat or analyzer
Supports research + experiments
""")

# ---------- ABOUT ----------
elif page == "About":
    st.title("👤 About")

    st.markdown("""
Mr. Mohan K is a researcher in Biotechnology, currently pursuing a PhD at the Vellore Institute of Technology.
His work focuses on AI-driven drug discovery, computational biology, and intelligent research systems.
He is building advanced platforms that combine artificial intelligence with biotechnology to accelerate scientific innovation.

### 📞 Contact
Phone: 9361245583
Email: mohanraj50115@gmail.com
""")

AI Research Assistant System
Supports RAG, multi-agent reasoning
Future: drug discovery AI
""")

In [ ]:
!pip install streamlit -q
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

!pkill streamlit || echo "No previous Streamlit process"
import time, subprocess

streamlit_proc = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])
time.sleep(8)

# Create tunnel
!cloudflared tunnel --url http://localhost:8501 --no-autoupdate



Selecting previously unselected package cloudflared.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.3.0) ...
Setting up cloudflared (2026.3.0) ...
Processing triggers for man-db (2.10.2-1) ...
No previous Streamlit process
2026-04-27T12:42:30Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-04-27T12:42:30Z INF Requesting new quick Tunnel on trycloudflare